# ArduPilot AI: Unsloth Fine-Tuning Pipeline (RTX 4050 6GB Optimized)

This notebook is built exactly from the official Unsloth Documentation to fine-tune `Qwen2.5-3B` on your programmatic ArduPilot command dataset. 
It is strictly configured for 4-bit quantization and extreme VRAM optimization so you don't OOM (crash) your 6GB card if you leave it running overnight.

### 1. Install Dependencies
Run this block **ONLY IF** you have not already installed `unsloth` in your `ardupilot_ai` conda environment.

In [ ]:
# !pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121
# !pip install "unsloth[cu121-ampere-torch220] @ git+https://github.com/unslothai/unsloth.git"
# !pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes datasets

### 2. Load the Model (4-Bit)
Loading the model in 4-bit allows a 3-Billion parameter model to fit easily in 6GB of VRAM.

In [3]:
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048 # Safe context window for 6GB VRAM
dtype = None # Auto detection
load_in_4bit = True # CRITICAL for RTX 4050

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-3B-Instruct", # Unsloth's optimized weights
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

ModuleNotFoundError: No module named 'huggingface_hub'

### 3. Configure LoRA Modifiers
This attaches trainable adapter weights to the model so we aren't training the entire neural network from scratch.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, # Target size
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth", # Saves 30% VRAM
    random_state = 3407,
)

### 4. Load & Format Your Dataset
Unsloth uses `get_chat_template` to automatically translate your `.jsonl` ChatML strings (which we generated earlier) into the exact native tokens Qwen uses. 

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

# Pre-format the tokenizer for Qwen
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "qwen-2.5", # Official Qwen syntax mapping
)

def formatting_prompts_func(examples):
    convos = examples["messages"]
    texts = [tokenizer.apply_chat_template(convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return { "text" : texts }

DATASET_PATH = "../scripts/qwen_finetune_dataset.jsonl"
print(f"Loading dataset from {DATASET_PATH}...")

dataset = load_dataset("json", data_files=DATASET_PATH, split="train")
dataset = dataset.map(formatting_prompts_func, batched = True)

### 5. Setup the SFTTrainer & Start Training Over-night
Configured for extremely low batch sizes to protect the 4050 GPU.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        # Setting learning_rate dynamically over an Epoch
        num_train_epochs = 1, # Run for 1 full pass over the 2,150 datasets (takes maybe 1-2 hours on a 4050)
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 10,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

# START TRAINING 
trainer_stats = trainer.train()

### 6. Export directly to Ollama!
Once training finishes, this block compiles the AI into a `.gguf` file that you can plug directly into your `config.py` in the ArduPilot Backend!

In [ ]:
# Export to Ollama compatible GGUF 
model.save_pretrained_gguf("ardupilot_ai_finetuned", tokenizer, quantization_method = "q4_k_m")

print("Training Complete! Your .gguf model has been exported to the ardupilot_ai_finetuned folder!")